# Notebook used  to experiment with different models

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
np.set_printoptions(precision=4)
np.set_printoptions(suppress=True)
from sklearn.metrics import  mean_pinball_loss

In [ ]:
!python3 -m pip install tensorflow_addons

In [ ]:
df = pd.read_csv("../traces/traces_persona.csv",parse_dates=[0])[:].reset_index()
diff = [df["reconsile_dates"][i] - df["reconsile_dates"][i-1] for i in range(1,len(df["reconsile_dates"]))]
diff_df = pd.DataFrame({"diff": diff, "sec": [x.total_seconds() for x in diff]}) 

endtrain  =  int(len(diff_df) * 0.8)
startTestIndex = endtrain
endtestIndex =  len(diff_df)-1

train_data = diff_df["sec"][:endtrain]
test_data = diff_df["sec"][endtrain:]

In [ ]:
df,diff_df


In [ ]:
df.describe(),diff_df.describe()

In [ ]:
df_stationarityTest = adfuller(df['reconsile_dates'], autolag='AIC')
print("P-value: ", df_stationarityTest[1])

In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf
pacf = plot_pacf(df['reconsile_dates'], lags=25)

# only use 2 long wide interval?

In [ ]:
plt.boxplot([diff_df["sec"]])

In [ ]:
_ =  plt.hist(diff_df["sec"],bins=200)

# choosing loss function

In [ ]:
# as defined in the scikit-learn  docs
# sklearn.metrics.mean_pinball_loss
def pinball_loss(y_true,y_pred, alpha=0.25):
    diff = y_true - y_pred
    sign = int(diff >= 0)
    loss = alpha * sign  * diff - (1 - alpha) * (1 - sign) * diff
    return loss

def total_perside_piball_loss(y_true_array,y_pred_array, alpha=0.25):
    assert len(y_pred_array) == len(y_true_array)
    underEstimate = []
    overEstimate = []
    totalEstimate = []
    for y_true,y_pred in zip(y_true_array,y_pred_array):

        diff = y_true - y_pred
        sign = int(diff >= 0)
        
        score = pinball_loss(y_true=y_true, y_pred=y_pred,alpha=alpha)
        
        if sign == 0:
            overEstimate.append(score)
        else:
            underEstimate.append(score)
        totalEstimate.append(score)
    
    underEstimate_score = np.mean(underEstimate)
    overEstimate_score = np.mean(overEstimate)
    ##print(totalEstimate)
    total_score = np.mean(totalEstimate)
    
    print(f"there are {len(underEstimate)} underEstimates with mean {underEstimate_score} ")
    print(f"there are {len(overEstimate)} overEstimates with mean {overEstimate_score}")
    print(f"total mean absolute pinball score {total_score}")
    return (underEstimate_score,overEstimate_score,total_score)

def plotDifferencepredictions(pred,testdata,startindex=None):
    #plt.rcParams["figure.figsize"] = (30, 10)


    
    pred = pd.DataFrame(pred)
    testdata = pd.DataFrame(testdata)

    if startindex is not None :
        pred.index = startindex
        testdata.index = startindex
    
        

    f, ax = plt.subplots() 
    f.set_figwidth(25)
    f.set_figheight(10)
    ax.plot(pred,'yo',label="prediction")
    ax.plot(testdata,'o', color='red',label="real")
    #plt.plot(pred,'yo',label="prediction")
    #plt.plot(testdata,'o', color='red',label="real")
    plt.legend(fontsize="xx-large")
    plt.show()





In [ ]:
alpha = 0.25 
x = np.concatenate([np.arange(-5,0,0.1), np.arange(0,5.1,0.1)])
y = [pinball_loss(0,i,alpha) for  i in x]
plt.title("Pinball loss function with α=0.25")
plt.ylabel("Error score")
plt.xlabel("Time difference in seconds")
plt.vlines(x=-2, ymin=0, ymax=0.5, colors='green',linestyles='dashed')
plt.hlines(y=0.5, xmin=-2, xmax=-5, colors='green',linestyles='dashed')

plt.vlines(x=2, ymin=0, ymax=1.5, colors='r',linestyles='dashed')
plt.hlines(y=1.5, xmin=2, xmax=-5, colors='r',linestyles='dashed')

plt.ylim(0,2.5)
plt.xlim(-5,5)

plt.plot(x,y)


## auto regression

In [ ]:
from statsmodels.tsa.ar_model import AutoReg
from pandas.plotting import lag_plot

In [ ]:
lag_plot(diff_df["sec"])

not really a lot of corrolation, no real line  visable

In [ ]:
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(diff_df["sec"])

as expected, since the times are constant, no real big  swings, by adding more lag, we don't gain anything 

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(diff_df["sec"], lags=50)

after 25 we see only small differences

In [ ]:
model = AutoReg(train_data, lags=10).fit()
print(model.summary())
pred = model.predict(start=startTestIndex, end=endtestIndex, dynamic=False)


In [ ]:
plotDifferencepredictions(pred,test_data)

In [ ]:
total_perside_piball_loss(test_data,pred,alpha=0.25)

instead of predicting all test results at a time, do 1 by 1 and relearn like  our framework can do

In [ ]:
pred = []
for i in range(startTestIndex,len(diff_df)):
    model = AutoReg(diff_df["sec"][:i], lags=10).fit()
    predict = model.predict(start=i+1, end=(i+1), dynamic=False)
    pred.append(predict[i+1])


In [ ]:
total_perside_piball_loss(test_data,pred,alpha=0.25)

In [ ]:
plotDifferencepredictions(pred,test_data)

as expected  we do not  see a lot of improvement

# arima

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import pmdarima as pm

In [ ]:
adf,pval,_,_,_,_ = adfuller(diff_df["sec"])
print(f"p value {pval} < 0.05 => stationairy")

choose what arima model

In [ ]:
model = pm.auto_arima(diff_df["sec"], start_p=1, start_q=1,
                      test='adf',       # use adftest to find optimal 'd'
                      max_p=10, max_q=10, # maximum p and q
                      m=1,              # frequency of series
                      d=None,           # let model determine 'd'
                      seasonal=False,   # No Seasonality
                      start_P=0, 
                      D=0, 
                      trace=True,
                      error_action='ignore',  
                      suppress_warnings=True, 
                      stepwise=True)


In [ ]:
model.summary()

In [ ]:
model = ARIMA(train_data, order=(2, 0, 0))

In [ ]:
result = model.fit()
result.summary()


In [ ]:
pred = result.predict(start=startTestIndex, end=endtestIndex)

In [ ]:
plotDifferencepredictions(pred,test_data)

In [ ]:
total_perside_piball_loss(test_data,pred,alpha=0.25)

In [ ]:
pred = []
for i in range(startTestIndex,len(diff_df)):
    model = ARIMA(diff_df["sec"][:i], order=(2, 0, 0)).fit()
    predict = model.predict(start=i+1, end=(i+1), dynamic=False)
    pred.append(predict[i+1])

In [ ]:
total_perside_piball_loss(test_data,pred,alpha=0.25)

In [ ]:
plotDifferencepredictions(pred,test_data)

# sarima

our data is  not really  seasonal so  skip this

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
model = SARIMAX(train_data, order=(2, 0, 0))

In [ ]:
result = model.fit()
result.summary()

In [ ]:
pred = result.predict(start=startTestIndex, end=endtestIndex)

In [ ]:
plotDifferencepredictions(pred,test_data)

In [ ]:
total_perside_piball_loss(test_data,pred,alpha=0.25)

predicting that far into the future is pritty bad

In [ ]:
pred = []
for i in range(startTestIndex,len(diff_df)):
    model = SARIMAX(diff_df["sec"][:i], order=(2, 0, 0)).fit()
    predict = model.predict(start=i+1, end=(i+1), dynamic=False)
    pred.append(predict[i+1])


In [ ]:
total_perside_piball_loss(test_data,pred,alpha=0.25)

model predicts pritty good, underestimates more, which is better for our framework

In [ ]:
plotDifferencepredictions(pred,test_data)

# Exponential smoothing

## simple  smoothing

In [ ]:
from statsmodels.tsa.api import  SimpleExpSmoothing

In [ ]:
model = SimpleExpSmoothing(train_data, initialization_method="estimated")
result = model.fit()

In [ ]:
pred =  result.predict(start=startTestIndex, end=endtestIndex)

In [ ]:
plotDifferencepredictions(pred,test_data)

In [ ]:
total_perside_piball_loss(test_data,pred,alpha=0.25)

In [ ]:
pred = []
for i in range(startTestIndex,len(diff_df)):
    model = SimpleExpSmoothing(diff_df["sec"][:i], initialization_method="estimated").fit()
    predict = model.predict(start=i+1, end=(i+1))
    pred.append(predict[i+1])

In [ ]:
total_perside_piball_loss(test_data,pred,alpha=0.25)

In [ ]:
plotDifferencepredictions(pred,test_data)

##  Holt

In [ ]:
from statsmodels.tsa.api import  Holt

In [ ]:
model = Holt(train_data, initialization_method="estimated")
result = model.fit()

In [ ]:
pred =  result.predict(start=startTestIndex, end=endtestIndex)

In [ ]:
plotDifferencepredictions(pred,test_data)

In [ ]:
total_perside_piball_loss(test_data,pred,alpha=0.25)

In [ ]:
pred = []
for i in range(startTestIndex,len(diff_df)):
    model = Holt(diff_df["sec"][:i], initialization_method="estimated").fit()
    predict = model.predict(start=i+1, end=(i+1))
    pred.append(predict[i+1])

In [ ]:
total_perside_piball_loss(test_data,pred,alpha=0.25)

In [ ]:
plotDifferencepredictions(pred,test_data)

# holtz winter

In [ ]:
from statsmodels.tsa.api import ExponentialSmoothing



In [ ]:
data  =  [5,5.05,5.1,4.68,4.88,4.99,5,5.12,5.42,5.19]
SimpleExpSmoothing(data, initialization_method="estimated").fit().predict(start=len(data), end=len(data))

In [ ]:
data  =  [5,5.05,5.1,4.68,4.88,4.99,5,5.12,5.42,5.19]
ExponentialSmoothing(data, initialization_method="estimated").fit().predict(start=len(data), end=len(data))

In [ ]:
model = ExponentialSmoothing(train_data, initialization_method="estimated")
result = model.fit()
pred =  result.predict(start=startTestIndex, end=endtestIndex)

In [ ]:
plotDifferencepredictions(pred,test_data)

In [ ]:
total_perside_piball_loss(test_data,pred,alpha=0.25)

In [ ]:
pred = []
for i in range(startTestIndex,len(diff_df)):
    model = ExponentialSmoothing(diff_df["sec"][:i], initialization_method="estimated").fit()
    predict = model.predict(start=i+1, end=(i+1))
    pred.append(predict[i+1])

In [ ]:
total_perside_piball_loss(test_data,pred,alpha=0.25)

In [ ]:
plotDifferencepredictions(pred,test_data)

# LSTM

In [ ]:
#https://medium.com/@nutanbhogendrasharma/simple-sequence-prediction-with-lstm-69ff0f4d57cd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_addons as tfa

def splitSequence(seq, n_steps):
    
    X = []
    y = []

    stopindex = len(seq)-n_steps
    
    for i in range(stopindex):
        #get the last index
        lastIndex = i + n_steps
        
        #Create input and output sequence
        seq_X, seq_y = seq[i:lastIndex], seq[lastIndex]
        
        #append seq_X, seq_y in X and y list
        X.append(seq_X)
        y.append(seq_y)     #Convert X and y into numpy array
    X = np.array(X)
    y = np.array(y)
    
    return X,y 



In [ ]:
n_steps  = 20

#  TODO rescale /normalize data ?

X,y = splitSequence(train_data,n_steps)

In [ ]:
# reshape from [samples, timesteps] into [samples, timesteps, features]n_features = 1
n_features = 1
X = X.reshape((X.shape[0], X.shape[1], n_features))
print(X[:2])

In [ ]:
model = tf.keras.Sequential()
model.add(layers.LSTM(30,return_sequences=True ,activation='relu', input_shape=(n_steps, n_features)))
model.add(layers.LSTM(units=10, return_sequences=True))
model.add(layers.LSTM(units=10))
model.add(layers.Dense(1))

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(0.01), loss=tfa.losses.PinballLoss(tau=.25), metrics=['mean_absolute_error'])

In [ ]:
model.fit(X, y, epochs=200, verbose=1)

In [ ]:
pred = []
for i in range(startTestIndex,len(diff_df)):

    hist = np.array(diff_df["sec"][i-n_steps:i]).reshape((1, n_steps, n_features))
    predict = model.predict(hist, verbose=1)
    pred.append(predict[0][0])

In [ ]:
total_perside_piball_loss(test_data,pred,alpha=0.25)

In [ ]:
total_perside_piball_loss(test_data,pred,alpha=0.25)

In [ ]:
plotDifferencepredictions(pred,test_data,test_data.index)

# SERVERTESTS

In [ ]:
from collections import deque
import requests
from datetime import datetime as dt, timezone, timedelta
import time

In [ ]:
def predictServer(function,df,bufferlen=50):
    q =  deque(maxlen=bufferlen)
    pred  = []

    for i in range(len( df['reconsile_dates'])-1):
        q.append(df['reconsile_dates'][i])
        history = [x.strftime('%Y-%m-%dT%H:%M:%S.%fZ') for x in q]
        r = requests.post('http://127.0.0.1:5000/prediction',json= {"history":history, "function": function})
        predictionstr  = r.json()["prediction"]
        predictiondate =  dt.strptime(predictionstr, '%Y-%m-%dT%H:%M:%S.%fZ')
        #print(predictiondate,df['reconsile_dates'][i+1].replace(tzinfo=None))
        secdif = (predictiondate - df['reconsile_dates'][i+1].replace(tzinfo=None)).total_seconds()
        pred.append(secdif)
        #time.sleep(2)
    
    #total_perside_piball_loss(np.zeros(len(pred)),pred)
    return pred



In [ ]:

#        load back mem                       predicted                            CURRENT
#               | ____TIMEBEFOREPREDICTEDMs________|_________GRACEPERIODMs________|
#                                                                                          


def calculateStatistics(timediffs, TIMEBEFOREPREDICTEDMs=0.5, GRACEPERIODMs=0.5, debug=True ):

    missesBeforeTIMEBEFOREPREDICTEDMs =0
    missesAfterGRACEPERIODMs  =  0
    caughtTIMEBEFOREPREDICTEDMs =  0
    caughtGRACEPERIODMs = 0
    for timedif in timediffs:
        if timedif < -TIMEBEFOREPREDICTEDMs:
            missesBeforeTIMEBEFOREPREDICTEDMs+=1
        elif timedif > GRACEPERIODMs:
            missesAfterGRACEPERIODMs+=1
        elif timedif <  0:
            caughtTIMEBEFOREPREDICTEDMs+=1
        elif timedif >  0:
            caughtGRACEPERIODMs+=1
    
    accuracy = (caughtGRACEPERIODMs+caughtTIMEBEFOREPREDICTEDMs)/len(timediffs)
    if not debug:
        return  accuracy

    print(f"        load back mem                       predicted                            CURRENT")
    print(f"               | ____TIMEBEFOREPREDICTEDMs________|_________GRACEPERIODMs________|")
    print(f"    {missesBeforeTIMEBEFOREPREDICTEDMs}                         {caughtTIMEBEFOREPREDICTEDMs}                              {caughtGRACEPERIODMs}                          {missesAfterGRACEPERIODMs}")
    print(f"accuracy= {accuracy}")
    return accuracy





In [ ]:
pred  =  predictServer("autoReg",df,bufferlen=50)

In [ ]:
plotDifferencepredictions(pred,np.zeros(len(pred)))


In [ ]:
lowerb, upperbound =  np.percentile(pred,[1,99])
perclist = [x for x in pred if lowerb < x <  upperbound]

plotDifferencepredictions(perclist,np.zeros(len(perclist)))

In [ ]:
calculateStatistics(pred)

ARIMA

In [ ]:
pred  =  predictServer("ARIMA",bufferlen=50)
plotDifferencepredictions(pred,np.zeros(len(pred)))

In [ ]:
lowerb, upperbound =  np.percentile(pred,[1,99])
perclist = [x for x in pred if lowerb < x <  upperbound]

plotDifferencepredictions(perclist,np.zeros(len(perclist)))

In [ ]:
calculateStatistics(pred)

SES

In [ ]:
pred  =  predictServer("SES",bufferlen=50)
plotDifferencepredictions(pred,np.zeros(len(pred)))

In [ ]:
lowerb, upperbound =  np.percentile(pred,[1,99])
perclist = [x for x in pred if lowerb < x <  upperbound]

plotDifferencepredictions(perclist,np.zeros(len(perclist)))

In [ ]:
calculateStatistics(pred)

Holt

In [ ]:
pred  =  predictServer("Holt",bufferlen=50)
plotDifferencepredictions(pred,np.zeros(len(pred)))

In [ ]:
lowerb, upperbound =  np.percentile(pred,[1,99])
perclist = [x for x in pred if lowerb < x <  upperbound]

plotDifferencepredictions(perclist,np.zeros(len(perclist)))

In [ ]:
calculateStatistics(pred)

winter

In [ ]:
pred  =  predictServer("Winter",bufferlen=50)
plotDifferencepredictions(pred,np.zeros(len(pred)))

In [ ]:
lowerb, upperbound =  np.percentile(pred,[1,99])
perclist = [x for x in pred if lowerb < x <  upperbound]

plotDifferencepredictions(perclist,np.zeros(len(perclist)))

In [ ]:
calculateStatistics(pred)

In [ ]:
n_steps  = 20
n_features = 1
def perdictserverLSTM(diff_df):

    endtrain  =  int(len(diff_df) * 0.8)
    startTestIndex = endtrain
    endtestIndex =  len(diff_df)-1

    train_data = diff_df["sec"][:endtrain]
    test_data = diff_df["sec"][endtrain:]

    X,y = splitSequence(train_data,n_steps)


    model = tf.keras.Sequential()
    model.add(layers.LSTM(30,return_sequences=True ,activation='relu', input_shape=(n_steps, n_features)))
    model.add(layers.LSTM(units=10, return_sequences=True))
    model.add(layers.LSTM(units=10))
    model.add(layers.Dense(1))
    model.compile(optimizer=tf.keras.optimizers.Adam(0.01), loss=tfa.losses.PinballLoss(tau=.25), metrics=['mean_absolute_error'])
    model.fit(X, y, epochs=200, verbose=1)


    pred = []
    for i in range(startTestIndex,len(diff_df)):

        hist = np.array(diff_df["sec"][i-n_steps:i]).reshape((1, n_steps, n_features))
        predict = model.predict(hist, verbose=1)
        preddiff = predict[0][0] - diff_df["sec"][i]
        pred.append(preddiff)
    return pred



Relationship  between length of startup  and  grace interval and the accurcay

In [ ]:
predAutoreg  =  predictServer("autoReg",df,bufferlen=50)
predArima  =  predictServer("ARIMA",df,bufferlen=50)
predSES  =  predictServer("SES",df, bufferlen=50)
predHolt  =  predictServer("Holt",df, bufferlen=50)
#predWinter  =  predictServer("Winter",bufferlen=50)

In [ ]:
predLstm  = perdictserverLSTM(diff_df)

In [ ]:
metrics = {
    "reg" : [],
    "arima" :  [],
    "ses" : [],
    "holt":[],
    "winter":[],
    "lstm":[]

}
xvalues = list(range(200,-1,-1))
for i in xvalues:

    metrics["reg"].append( calculateStatistics(predAutoreg,TIMEBEFOREPREDICTEDMs=i/1000,GRACEPERIODMs=i/1000,debug=False))
    metrics["arima"].append(calculateStatistics(predArima,TIMEBEFOREPREDICTEDMs=i/1000,GRACEPERIODMs=i/1000,debug=False))
    metrics["ses"].append(calculateStatistics(predSES,TIMEBEFOREPREDICTEDMs=i/1000,GRACEPERIODMs=i/1000,debug=False))
    metrics["holt"].append(calculateStatistics(predHolt,TIMEBEFOREPREDICTEDMs=i/1000,GRACEPERIODMs=i/1000,debug=False))
    #metrics["winter"].append(calculateStatistics(predWinter,TIMEBEFOREPREDICTEDMs=i/1000,GRACEPERIODMs=i/1000,debug=False))
    metrics["lstm"].append(calculateStatistics(predLstm,TIMEBEFOREPREDICTEDMs=i/1000,GRACEPERIODMs=i/1000,debug=False))

In [ ]:
plt.title("Model performance with varying start-up and grace intervals")
plt.ylabel("Accuracy score")
plt.xlabel("Start-up and grace interval periods in milliseconds")
plt.plot(xvalues,metrics["reg"],label = "AR")
plt.plot(xvalues,metrics["arima"],label = "ARIMA")
plt.plot(xvalues,metrics["ses"],label = "SES")
plt.plot(xvalues,metrics["holt"],label = "Holt")
#plt.plot(xvalues,metrics["winter"],label = "Winter")
plt.plot(xvalues,metrics["lstm"],label = "LSTM")
ax = plt.gca()
ax.invert_xaxis()
plt.legend()
plt.show()





relation between buffer len  and  accuracy

In [ ]:

metrics = {
    "reg" : [],
    "arima" :  [],
    "ses" : [],
    "holt":[],
    "winter":[],
    "lstm":[]

}
xvalues = list(range(25,70,5))
for i in xvalues:
    print(i)


    predAutoreg  =  predictServer("autoReg",df,bufferlen=i)
    predArima  =  predictServer("ARIMA",df,bufferlen=i)
    predSES  =  predictServer("SES",df,bufferlen=i)
    predHolt  =  predictServer("Holt",df,bufferlen=i)
    #predWinter  =  predictServer("Winter",bufferlen=i)

    metrics["reg"].append( calculateStatistics(predAutoreg,TIMEBEFOREPREDICTEDMs=0.2,GRACEPERIODMs=0.2,debug=False))
    metrics["arima"].append(calculateStatistics(predArima,TIMEBEFOREPREDICTEDMs=0.2,GRACEPERIODMs=0.2,debug=False))
    metrics["ses"].append(calculateStatistics(predSES,TIMEBEFOREPREDICTEDMs=0.2,GRACEPERIODMs=0.2,debug=False))
    metrics["holt"].append(calculateStatistics(predHolt,TIMEBEFOREPREDICTEDMs=0.2,GRACEPERIODMs=0.2,debug=False))
    #metrics["winter"].append(calculateStatistics(predWinter,TIMEBEFOREPREDICTEDMs=0.2,GRACEPERIODMs=0.2,debug=False))
    






In [ ]:
plt.title("Model performance with varying amounts of reconcile history")
plt.ylabel("Accuracy score")
plt.xlabel("amount of training data")
plt.plot(xvalues,metrics["reg"],label = "AR")
plt.plot(xvalues,metrics["arima"],label = "ARIMA")
plt.plot(xvalues,metrics["ses"],label = "SES")
plt.plot(xvalues,metrics["holt"],label = "Holt")
#plt.plot(xvalues,metrics["winter"],label = "Winter")
#plt.plot(xvalues,metrics["lstm"],label = "LSTM")
ax = plt.gca()
#ax.invert_xaxis()
plt.legend()
plt.show()


#model performance with   varying time intervals, just  add to it,  this way  noise is  still present

In [ ]:

metrics = {
    "reg" : [],
    "arima" :  [],
    "ses" : [],
    "holt":[],
    "winter":[],
    "lstm":[]

}
xvalues = list(range(5,65,5))
for i in xvalues:

    diffAdded = [x  + timedelta(seconds=i) for x in  diff]
    dfAdded = df.copy(deep=True)
    for t in  range(len(diffAdded)):
        dfAdded["reconsile_dates"][t+1] = dfAdded["reconsile_dates"][t]+ diffAdded[t]
    diff_dfAdded = pd.DataFrame({"diff": diffAdded, "sec": [x.total_seconds() for x in diffAdded]}) 
    print(i)

    predAutoreg  =  predictServer("autoReg",dfAdded,bufferlen=50)
    predArima  =  predictServer("ARIMA",dfAdded,bufferlen=50)
    predSES  =  predictServer("SES",dfAdded,bufferlen=50)
    predHolt  =  predictServer("Holt",dfAdded,bufferlen=50)
    #predWinter  =  predictServer("Winter",dfAdded,bufferlen=50)
    predLstm  = perdictserverLSTM(diff_dfAdded)

    metrics["reg"].append( calculateStatistics(predAutoreg,TIMEBEFOREPREDICTEDMs=0.2,GRACEPERIODMs=0.2,debug=False))
    metrics["arima"].append(calculateStatistics(predArima,TIMEBEFOREPREDICTEDMs=0.2,GRACEPERIODMs=0.2,debug=False))
    metrics["ses"].append(calculateStatistics(predSES,TIMEBEFOREPREDICTEDMs=0.2,GRACEPERIODMs=0.2,debug=False))
    metrics["holt"].append(calculateStatistics(predHolt,TIMEBEFOREPREDICTEDMs=0.2,GRACEPERIODMs=0.2,debug=False))
    #metrics["winter"].append(calculateStatistics(predWinter,TIMEBEFOREPREDICTEDMs=0.2,GRACEPERIODMs=0.2,debug=False))
    metrics["lstm"].append(calculateStatistics(predLstm,TIMEBEFOREPREDICTEDMs=0.2,GRACEPERIODMs=0.2,debug=False))
    


In [ ]:
plt.title("Model performance with varying reconcile intervals")
plt.ylabel("Accuracy score")
plt.xlabel("Reconcile interval in seconds")
plt.plot(xvalues,metrics["reg"],label = "AR")
plt.plot(xvalues,metrics["arima"],label = "ARIMA")
plt.plot(xvalues,metrics["ses"],label = "SES")
plt.plot(xvalues,metrics["holt"],label = "Holt")
#plt.plot(xvalues,metrics["winter"],label = "Winter")
plt.plot(xvalues,metrics["lstm"],label = "LSTM")
plt.xticks(xvalues, xvalues)
plt.legend()
plt.show()

